# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata and print basic information
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This step inspects all record sets defined in the dataset metadata, lists their `@id`s, and, for each record set, prints out field/column `@id`s and names.

In [ ]:
# List all record sets and their fields/columns by @id

record_sets = []

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print("Record sets found in the dataset:")
    for rs in metadata.recordSet:
        if hasattr(rs, '@id'):
            record_sets.append(rs['@id'])
            print(f"- RecordSet @id: {rs['@id']} (name: {getattr(rs, 'name', 'N/A')})")
            # List fields/columns for this record set
            if hasattr(rs, 'field'):
                print("  Fields/columns:")
                for fld in rs.field:
                    print(f"    Field @id: {fld['@id']} (name: {getattr(fld, 'name', 'N/A')})")
else:
    # As a fallback, try to get record set ids from the dataset itself
    print("No record sets listed in metadata. Detecting available record set ids from dataset...")
    try:
        rs_ids = dataset.record_set_ids()
        record_sets = rs_ids
        for rs_id in rs_ids:
            print(f"- RecordSet @id: {rs_id}")
    except Exception as e:
        print("Unable to determine record sets from dataset.", e)

# Optionally, preview records from the first record set
if record_sets:
    first_rs_id = record_sets[0]
    print(f"\nPreview of sample records from RecordSet @id: {first_rs_id}")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        if i >= 3:
            break
        print(rec)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}
print("\nExtracting tabular data for all record sets...")

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nDataFrame for RecordSet @id: {rs_id} (shape: {df.shape})")
        print("Columns:", df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for RecordSet @id: {rs_id}")

# Select the primary record set for further analysis
main_record_set_id = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section demonstrates operations such as removing outliers, normalizing fields, and grouping records by key demographic attributes.

All fields, columns, and record sets referenced by their `@id`.

In [ ]:
# Pick a numeric field by @id from the main record set for analysis, e.g., GAD-7 total score
# Example: let's assume the column @id is 'cr:GAD7_total_score' (adjust as real @id)

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Identify numeric columns for EDA
    numeric_cols = [col for col in df.columns if 'score' in col or 'age' in col or df[col].dtype in ['int64', 'float64']]
    print("Numeric columns detected:", numeric_cols)

    # Pick one for demonstration (default: first numeric col)
    numeric_field_id = numeric_cols[0] if numeric_cols else df.columns[0]

    # Filter values above a threshold (e.g., mean + 1 std or arbitrary value)
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by categorical field, e.g., 'cr:gender' if available
    group_fields = [col for col in df.columns if 'gender' in col or 'education' in col or 'marital' in col]
    group_field_id = group_fields[0] if group_fields else None

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean values of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No main record set dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot a histogram of the selected numeric field, and, if possible, boxplots grouped by a demographic field (all fields referenced using their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]

    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot grouped by group_field_id if available
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
We have successfully loaded and explored the FAIR² Mental Health Survey dataset using the `mlcroissant` library. Key findings include reviewing available record sets and fields by their unique `@id`, extracting tabular data to pandas DataFrames, filtering and normalizing scores (such as GAD-7, PHQ-9, or other numeric mental health indicators), and visualizing relationships such as score distribution by gender or education.

Further analysis can focus on detailed health outcomes, demographic patterns, and AI-ready data processing, leveraging the dataset's structure and standards.